
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 4L - Window Aggregation in Spark Structured Streaming

In this lab, you'll work with stateful operations, sliding windows, and watermarks in Spark Structured Streaming. You'll analyze streams of order and status data to derive meaningful insights.

### Objectives
- Implement stateful aggregations and window operations
- Handle late data and state management
- Build real-time monitoring systems

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Setup and Data Sources

First, let's set up our streaming environment with the necessary data sources.

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Schemas are provided for you
orders_schema = StructType([
    StructField("customer_id", LongType(), True),
    StructField("notifications", StringType(), True),
    StructField("order_id", LongType(), True),
    StructField("order_timestamp", LongType(), True)
])

status_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("order_status", StringType(), True),
    StructField("status_timestamp", LongType(), True)
])

# Create status streaming DataFrame
status_stream = spark.readStream \
    .format("json") \
    .schema(status_schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("path", "/Volumes/dbacademy_retail/v01/retail-pipeline/status/stream_json") \
    .load()

# Create orders streaming DataFrame
orders_stream = spark.readStream \
    .format("json") \
    .schema(orders_schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("path", "/Volumes/dbacademy_retail/v01/retail-pipeline/orders/stream_json") \
    .load()

# Add event_time column to status stream
status_events = status_stream \
    .withColumn("event_time", from_unixtime(col("status_timestamp")).cast("timestamp"))

# Verify streams are set up correctly
print(f"orders_stream is streaming: {orders_stream.isStreaming}")
print(f"status_stream is streaming: {status_stream.isStreaming}")

orders_stream is streaming: True
status_stream is streaming: True


## B. Stateful Operations

Let's explore stateful operations that maintain state across micro-batches.

In [0]:
%python

# First, clean up any existing queries with the same names
for query in spark.streams.active:
    if query.name in ["status_counts", "customer_counts"]:
        query.stop()

# Count orders by status (stateful aggregation)
status_counts = status_stream \
    .groupBy("order_status") \
    .count() \
    .orderBy(col("count").desc())

# Count orders by customer (stateful aggregation)
customer_counts = orders_stream \
    .groupBy("customer_id") \
    .count() \
    .orderBy(col("count").desc())

# Write status counts to memory
status_query = status_counts.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("status_counts") \
    .start()

# Write customer counts to memory
customer_query = customer_counts.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("customer_counts") \
    .start()

Now you can query these tables to see the results:

In [0]:
-- Query the in-memory table to see status counts
SELECT * FROM status_counts

order_status,count
placed,174
preparing,109
on the way,107
delivered,94
return requested,15
return picked up,9
canceled,8
return processed,8
return canceled,6
reported shipping error,6


In [0]:
-- Query the in-memory table to see customer counts
select * from customer_counts

## C. Sliding Window Operations
In this section, you'll implement sliding window aggregations on the streaming data.


In [0]:
%python

# First, clean up any existing queries with the same name
for query in spark.streams.active:
    if query.name == "sliding_windows":
        query.stop()

# Create sliding window aggregation
sliding_window_counts = status_events \
    .groupBy(
        window(col("event_time"), "3 minutes", "1 minute"),
        col("order_status")
    ) \
    .count()

# Write sliding window counts to memory
sliding_window_query = sliding_window_counts.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("sliding_windows") \
    .start()

You can query the sliding window results:

In [0]:
-- Query the sliding window results
SELECT 
  window.start as window_start,
  window.end as window_end,
  order_status,
  count
FROM sliding_windows
ORDER BY window_start, order_status

window_start,window_end,order_status,count
2021-12-25T00:26:00Z,2021-12-25T00:29:00Z,placed,1
2021-12-25T00:27:00Z,2021-12-25T00:30:00Z,placed,1
2021-12-25T00:28:00Z,2021-12-25T00:31:00Z,placed,1
2021-12-25T00:33:00Z,2021-12-25T00:36:00Z,placed,1
2021-12-25T00:34:00Z,2021-12-25T00:37:00Z,placed,1
2021-12-25T00:35:00Z,2021-12-25T00:38:00Z,placed,1
2021-12-25T01:12:00Z,2021-12-25T01:15:00Z,placed,1
2021-12-25T01:13:00Z,2021-12-25T01:16:00Z,placed,1
2021-12-25T01:14:00Z,2021-12-25T01:17:00Z,placed,1
2021-12-25T01:32:00Z,2021-12-25T01:35:00Z,placed,1


## D. Late Data Handling with Watermarks
Now, let's explore how to handle late-arriving data using watermarks.

In [0]:
%python

# First, clean up any existing queries with the same names
for query in spark.streams.active:
    if query.name in ["windowed_with_watermark", "joined_with_watermark"]:
        query.stop()

# Add watermark to status events
status_with_watermark = status_events \
    .withWatermark("event_time", "5 minutes")

# Windows with watermark
watermarked_windows = status_with_watermark \
    .groupBy(
        window(col("event_time"), "3 minutes", "1 minute"),
        col("order_status")
    ) \
    .count()

# Write to memory
watermark_query = watermarked_windows.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("windowed_with_watermark") \
    .start()

# Prepare orders stream with event_time for join
orders_with_time = orders_stream \
    .withColumn("order_time", from_unixtime(col("order_timestamp")).cast("timestamp")) \
    .withWatermark("order_time", "5 minutes")

# Join with watermarks to limit state
watermarked_join = orders_with_time \
    .join(
        status_with_watermark,
        "order_id",
        "inner"
    )

# Write to memory
join_query = watermarked_join.writeStream \
    .format("memory") \
    .outputMode("append") \
    .queryName("joined_with_watermark") \
    .start()

Query the results:

In [0]:
SELECT 
  window.start as window_start,
  window.end as window_end,
  order_status,
  count
FROM windowed_with_watermark
ORDER BY window_start, order_status

window_start,window_end,order_status,count
2021-12-25T00:26:00Z,2021-12-25T00:29:00Z,placed,1
2021-12-25T00:27:00Z,2021-12-25T00:30:00Z,placed,1
2021-12-25T00:28:00Z,2021-12-25T00:31:00Z,placed,1
2021-12-25T00:33:00Z,2021-12-25T00:36:00Z,placed,1
2021-12-25T00:34:00Z,2021-12-25T00:37:00Z,placed,1
2021-12-25T00:35:00Z,2021-12-25T00:38:00Z,placed,1
2021-12-25T01:12:00Z,2021-12-25T01:15:00Z,placed,1
2021-12-25T01:13:00Z,2021-12-25T01:16:00Z,placed,1
2021-12-25T01:14:00Z,2021-12-25T01:17:00Z,placed,1
2021-12-25T01:32:00Z,2021-12-25T01:35:00Z,placed,1


In [0]:
SELECT 
  order_id, 
  customer_id, 
  order_status,
  notifications,
  order_time,
  event_time as status_time
FROM joined_with_watermark
LIMIT 20

order_id,customer_id,order_status,notifications,order_time,status_time
75167,23242,placed,N,2021-12-26T16:05:21Z,2021-12-26T16:05:21Z
75190,23836,placed,N,2021-12-27T17:15:22Z,2021-12-27T17:15:22Z
75232,23273,placed,Y,2021-12-29T05:15:06Z,2021-12-29T05:15:06Z
75264,23312,placed,Y,2021-12-30T17:55:12Z,2021-12-30T17:55:12Z
75167,23242,preparing,N,2021-12-26T16:05:21Z,2021-12-28T10:04:15Z
75167,23242,on the way,N,2021-12-26T16:05:21Z,2021-12-30T22:51:44Z
75167,23242,delivered,N,2021-12-26T16:05:21Z,2021-12-30T21:23:59Z
75190,23836,preparing,N,2021-12-27T17:15:22Z,2021-12-30T02:26:10Z
75190,23836,on the way,N,2021-12-27T17:15:22Z,2021-12-29T14:07:04Z
75190,23836,delivered,N,2021-12-27T17:15:22Z,2021-12-28T23:53:36Z


Run the cell below to stop the active streaming queries.

In [0]:
%python
for query in spark.streams.active:
    query.stop()


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
